### Specialist Few-Shot Model

In [30]:
# Import necessary libraries
import os
import sys
# Add the utils folder to the system path so we can import custom modules
sys.path.append(os.path.abspath(os.path.join("../..")))
import matplotlib.pyplot as plt
import torch
import numpy as np
import torch.nn as nn
import pandas as pd
from utils.reproducibility_utils import seed_everything
from utils.data_loading import load_data, load_test_data, feature_names

# Set the plotting style for better-looking visualizations
plt.style.available
plt.style.use('seaborn-v0_8')

In [31]:
# Set random seed for reproducibility across runs
# This ensures that your results will be the same every time you run the notebook
seed_everything(42)  # for reproducibility

In [32]:
import glob
# -----------------------------
# 1. LOAD ALL TRAINING CSVs
# -----------------------------
train_path = "../../datasets_challenge/train"

train_dfs = []
for f in sorted(glob.glob(os.path.join(train_path, "*.csv"))):
    print("Loaded:", f)
    train_dfs.append(pd.read_csv(f))

# Merge everything into one dataframe
train_df = pd.concat(train_dfs, ignore_index=True)
print("Total samples:", len(train_df))

Loaded: ../../datasets_challenge/train\train_part_001.csv
Loaded: ../../datasets_challenge/train\train_part_002.csv
Loaded: ../../datasets_challenge/train\train_part_003.csv
Loaded: ../../datasets_challenge/train\train_part_004.csv
Loaded: ../../datasets_challenge/train\train_part_005.csv
Loaded: ../../datasets_challenge/train\train_part_006.csv
Loaded: ../../datasets_challenge/train\train_part_007.csv
Loaded: ../../datasets_challenge/train\train_part_008.csv
Loaded: ../../datasets_challenge/train\train_part_009.csv
Loaded: ../../datasets_challenge/train\train_part_010.csv
Loaded: ../../datasets_challenge/train\train_part_011.csv
Loaded: ../../datasets_challenge/train\train_part_012.csv
Loaded: ../../datasets_challenge/train\train_part_013.csv
Loaded: ../../datasets_challenge/train\train_part_014.csv
Loaded: ../../datasets_challenge/train\train_part_015.csv
Loaded: ../../datasets_challenge/train\train_part_016.csv
Loaded: ../../datasets_challenge/train\train_part_017.csv
Loaded: ../../

In [33]:
# -----------------------------
# 2. LOAD SIDESLIP DISTRIBUTION
# -----------------------------
dist_df = pd.read_csv("../../datasets_challenge/sideSlip_distribution.csv")

# few-shot bins
few_bins = dist_df[dist_df["band"] == "few"]["sideSlip"].tolist()
few_bins = [round(x, 3) for x in few_bins]

print("Few-shot bins:", few_bins[:20], " ...")
print("Number of few-shot bins:", len(few_bins))

Few-shot bins: [-14.9, -14.2, -14.0, -13.7, -12.7, -12.1, -12.0, -11.9, -11.7, -11.6, -11.5, -11.4, -11.3, -11.2, -11.1, -11.0, -10.9, -10.8, -10.7, -10.6]  ...
Number of few-shot bins: 151


In [34]:
# -----------------------------
# 3. FILTER INTO SPLITS
# -----------------------------

# Round sideSlip so it matches distribution CSV bin values
train_df["sideSlip_rounded"] = train_df["sideSlip"].round(3)

# A) Full dataset (nothing filtered)
df_A = train_df.copy()

# B) Tail dataset = ONLY rows falling in few-shot bins
df_B = train_df[ train_df["sideSlip_rounded"].isin(few_bins) ].copy()

# C) Curriculum dataset:
#    - Include ALL tail rows
#    - Include sampled many-shot rows (to maintain balance)

# Extract many-shot bins
many_bins = dist_df[dist_df["band"] == "many"]["sideSlip"].tolist()
many_bins = [round(x, 3) for x in many_bins]

df_many = train_df[train_df["sideSlip_rounded"].isin(many_bins)].copy()

# Sample many-shot at fraction (you can tune this)
SAMPLE_FRACTION = 0.10  # 10% of many-shot to mix with tail

df_many_sampled = df_many.sample(frac=SAMPLE_FRACTION, random_state=42)

# Curriculum = all few-shot + small many-shot sample
df_C = pd.concat([df_B, df_many_sampled], ignore_index=True)

print("\nDataset sizes:")
print("Model A (full):", len(df_A))
print("Model B (tail only):", len(df_B))
print("Model C (curriculum):", len(df_C))


Dataset sizes:
Model A (full): 275226
Model B (tail only): 422
Model C (curriculum): 14983


In [35]:
# Output folders
os.makedirs("data_model_A", exist_ok=True)
os.makedirs("data_model_B", exist_ok=True)
os.makedirs("data_model_C", exist_ok=True)

# Save as CSVs
df_A.to_csv("data_model_A/train.csv", index=False)
df_B.to_csv("data_model_B/train.csv", index=False)
df_C.to_csv("data_model_C/train.csv", index=False)

print("Saved dataset splits.")

Saved dataset splits.


In [36]:
FEW_MIN = -0.015
FEW_MAX =  0.015

def extract_few_shot_sequences(loader, low=FEW_MIN, high=FEW_MAX):
    X_few = []
    y_few = []

    for X_batch, y_batch in loader:
        mask = (y_batch >= low) & (y_batch <= high)
        if mask.any():
            X_few.append(X_batch[mask])
            y_few.append(y_batch[mask])

    if len(X_few) == 0:
        raise ValueError("No few-shot samples found! Adjust FEW_MIN/FEW_MAX.")

    X_few = torch.cat(X_few, dim=0)
    y_few = torch.cat(y_few, dim=0)

    print("Few-shot samples extracted:", len(y_few))
    return X_few, y_few

X_few, y_few = extract_few_shot_sequences(train_loader_A)


Few-shot samples extracted: 3780


In [37]:
oversample_factor = 30

X_few_over = X_few.repeat((oversample_factor, 1, 1))
y_few_over = y_few.repeat((oversample_factor,))

few_dataset = torch.utils.data.TensorDataset(X_few_over, y_few_over)
few_loader = torch.utils.data.DataLoader(few_dataset, batch_size=64, shuffle=True)

print("Few-shot oversampled dataset size:", len(few_dataset))


Few-shot oversampled dataset size: 113400


In [38]:
train_loader_A, val_loader_A, pipe_A = load_data(
    train_path="data_model_A",
    train_ratio=0.99,
    batchsize=64,
    columns_to_drop=["sideSlip","vyCG","time","longitude","latitude"],
    sequence_length=20,
    threshold=True,
    threshold_value=3.0,
    threshold_column="vxCG",
    target="sideSlip",
    scaling=True,
    seed=42,
)

train_loader_B, val_loader_B, pipe_B = load_data(
    train_path="data_model_B",
    train_ratio=None,  # no need for validation; train ONLY tail
    batchsize=64,
    columns_to_drop=["sideSlip","vyCG","time","longitude","latitude"],
    sequence_length=20,
    threshold=True,
    threshold_value=3.0,
    threshold_column="vxCG",
    target="sideSlip",
    scaling=True,
    seed=42,
)

train_loader_C, val_loader_C, pipe_C = load_data(
    train_path="data_model_C",
    train_ratio=0.95,  # curriculum should have validation
    batchsize=64,
    columns_to_drop=["sideSlip","vyCG","time","longitude","latitude"],
    sequence_length=20,
    threshold=True,
    threshold_value=3.0,
    threshold_column="vxCG",
    target="sideSlip",
    scaling=True,
    seed=42,
)

print("All dataloaders ready.")


data_model_A\train.csv
data_model_B\train.csv
data_model_C\train.csv
All dataloaders ready.


In [39]:
few_dataset = torch.utils.data.TensorDataset(X_few_over, y_few_over)
few_loader = torch.utils.data.DataLoader(few_dataset, batch_size=64, shuffle=True)

In [40]:
class RNN(nn.Module):
    """
    Recurrent Neural Network for sideslip angle prediction.
    
    Architecture:
    1. RNN Layer - Processes sequential input
    2. Fully Connected Layer - Transforms RNN output
    3. Output Layer - Produces final prediction
    
    Dropout layers are included to prevent overfitting.
    """
    def __init__(self, input_size, output_size=1, rnn_units=28, fc_units=28, 
                 activation_fct=nn.ReLU(), dropout=[0.0, 0, 0], rnn_nonlinearity='relu'):
        super(RNN, self).__init__()
        
        # RNN layer: processes the sequence and captures temporal dependencies
        self.rnn_layer = nn.RNN(input_size=input_size, hidden_size=rnn_units, 
                                num_layers=1, nonlinearity=rnn_nonlinearity, batch_first=True)
        
        # Fully connected layers for processing RNN output
        self.fc = nn.Linear(rnn_units, fc_units)
        self.output_layer = nn.Linear(fc_units, output_size)
        
        # Dropout layers to prevent overfitting (randomly zeros out neurons during training)
        self.dropout1 = nn.Dropout(dropout[0])  # Applied to input
        self.dropout2 = nn.Dropout(dropout[1])  # Applied after RNN
        self.dropout3 = nn.Dropout(dropout[2])  # Applied after FC layer
        
        self.activation_fct = activation_fct
 
    def forward(self, x):
        # x shape: (batch_size, sequence_length, input_size)
        x = self.dropout1(x)
        
        # Process sequence through RNN
        x, _ = self.rnn_layer(x)  # Output shape: (batch_size, sequence_length, rnn_units)
        
        # Take only the last time step's output (t0)
        x = x[:, -1, :]  # Shape: (batch_size, rnn_units)
        
        x = self.dropout2(x)
        x = self.activation_fct(self.fc(x))  # Apply fully connected layer with activation
        x = self.dropout3(x)
        x = self.output_layer(x)  # Final prediction
        
        return x

In [41]:
def train_rnn_model(model, train_loader, val_loader=None,
                    lr=3e-4, num_epochs=40, early_stop=8,
                    model_name="model"):
    
    device = next(model.parameters()).device
    criterion = nn.L1Loss()
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)

    best_loss = float("inf")
    best_epoch = 0

    for epoch in range(num_epochs):
        model.train()
        train_loss = 0

        for X, y in train_loader:
            X = X.to(device)
            y = y.to(device).reshape(-1, 1)

            optimizer.zero_grad()
            pred = model(X)
            loss = criterion(pred, y)
            loss.backward()
            optimizer.step()

            train_loss += loss.item()

        avg_train = train_loss / len(train_loader)

        # ---------------------- validation -----------------------
        if val_loader:
            model.eval()
            val_loss = 0

            with torch.no_grad():
                for Xv, yv in val_loader:
                    Xv = Xv.to(device)
                    yv = yv.to(device).reshape(-1, 1)
                    pred_val = model(Xv)
                    val_loss += criterion(pred_val, yv).item()

            avg_val = val_loss / len(val_loader)
            print(f"{model_name}: Epoch {epoch+1} | Train={avg_train:.5f} | Val={avg_val:.5f}")

            # save checkpoint
            if avg_val < best_loss:
                best_loss = avg_val
                best_epoch = epoch
                torch.save(model.state_dict(), f"{model_name}_best.pth")

            elif epoch - best_epoch > early_stop:
                print("Early stopping:", model_name)
                break

        else:
            print(f"{model_name}: Epoch {epoch+1} | Train={avg_train:.5f}")

    print(f"✓ {model_name} finished training. Best epoch={best_epoch+1}")


In [42]:
device = "cpu"

# generalist
model_A = RNN(input_size=X.shape[2]).to(device)
train_rnn_model(model_A, train_loader_A, val_loader_A, model_name="model_A")

# few-shot specialist
model_B = RNN(input_size=X_few.shape[2]).to(device)
train_rnn_model(model_B, few_loader, val_loader=None, model_name="model_B_fewshot")


model_A: Epoch 1 | Train=0.11467 | Val=0.03590
model_A: Epoch 2 | Train=0.01431 | Val=0.02961
model_A: Epoch 3 | Train=0.00927 | Val=0.01541
model_A: Epoch 4 | Train=0.00718 | Val=0.01377
model_A: Epoch 5 | Train=0.00607 | Val=0.01201
model_A: Epoch 6 | Train=0.00556 | Val=0.00843
model_A: Epoch 7 | Train=0.00553 | Val=0.00632
model_A: Epoch 8 | Train=0.00500 | Val=0.00624
model_A: Epoch 9 | Train=0.00481 | Val=0.00331
model_A: Epoch 10 | Train=0.00454 | Val=0.00843
model_A: Epoch 11 | Train=0.00438 | Val=0.02092
model_A: Epoch 12 | Train=0.00435 | Val=0.00853
model_A: Epoch 13 | Train=0.00401 | Val=0.01004
model_A: Epoch 14 | Train=0.00396 | Val=0.00942
model_A: Epoch 15 | Train=0.00385 | Val=0.00444
model_A: Epoch 16 | Train=0.00408 | Val=0.00978
model_A: Epoch 17 | Train=0.00361 | Val=0.00752
model_A: Epoch 18 | Train=0.00374 | Val=0.00385
Early stopping: model_A
✓ model_A finished training. Best epoch=9
model_B_fewshot: Epoch 1 | Train=0.01587
model_B_fewshot: Epoch 2 | Train=0.006

In [43]:
test_dataloader = load_test_data(
    test_path="../../datasets_challenge/test",
    batchsize=64,
    pipeline=pipe_A,
    load_anonymized_data=True
)


../../datasets_challenge/test\test_part_001.csv
../../datasets_challenge/test\test_part_002.csv
../../datasets_challenge/test\test_part_003.csv
../../datasets_challenge/test\test_part_004.csv
../../datasets_challenge/test\test_part_005.csv
../../datasets_challenge/test\test_part_006.csv
../../datasets_challenge/test\test_part_007.csv
../../datasets_challenge/test\test_part_008.csv
../../datasets_challenge/test\test_part_009.csv
../../datasets_challenge/test\test_part_010.csv
../../datasets_challenge/test\test_part_011.csv


In [46]:

model_A.eval()
model_B.eval()


RNN(
  (rnn_layer): RNN(23, 28, batch_first=True)
  (fc): Linear(in_features=28, out_features=28, bias=True)
  (output_layer): Linear(in_features=28, out_features=1, bias=True)
  (dropout1): Dropout(p=0.0, inplace=False)
  (dropout2): Dropout(p=0, inplace=False)
  (dropout3): Dropout(p=0, inplace=False)
  (activation_fct): ReLU()
)

In [47]:
def blend_predictions(pred_A, pred_B, alpha=0.04):
    """
    Smaller |pred_A| -> more weight to specialist model_B.
    """
    conf = torch.abs(pred_A)          # confidence low near 0

    w_B = torch.exp(-conf / alpha)    # high weight when conf small
    w_A = 1 - w_B

    return w_A * pred_A + w_B * pred_B


In [48]:
final_preds = []

with torch.no_grad():
    for X, _ in test_dataloader:
        X = X.to(device)

        pred_A = model_A(X)
        pred_B = model_B(X)

        blended = blend_predictions(pred_A, pred_B)
        final_preds.append(blended.cpu().numpy())

final_preds = np.concatenate(final_preds).reshape(-1)


In [49]:
df = pd.DataFrame({
    "predicted_sideslip": final_preds,
    "id": range(len(final_preds))
})

df.to_csv("predictions_fewshot_specialist.csv", index=False)
print("✓ Saved predictions_fewshot_specialist.csv")


✓ Saved predictions_fewshot_specialist.csv
